In [ ]:
# ==========================================
# 1. IMPORTS & TEMPLATES
# ==========================================
import getpass
import os
import xml.etree.ElementTree as ET
import requests
from requests.auth import HTTPBasicAuth

# SRU SOAP update request template
SOAP_TEMPLATE = """<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
  <soap:Body>
    <ucp:updateRequest xmlns:ucp="http://www.loc.gov/zing/srw/update/" xmlns:srw="http://www.loc.gov/zing/srw/" xmlns:diag="http://www.loc.gov/zing/srw/diagnostic/">
      <srw:version>1.0</srw:version>
      <ucp:action>{action}</ucp:action>
      <srw:record>
        <srw:recordPacking>xml</srw:recordPacking>
        <srw:recordSchema>MARC21-xml</srw:recordSchema>
        <srw:recordData>
          <record xmlns="http://www.loc.gov/MARC21/slim">
{marc_content}
          </record>
        </srw:recordData>
      </srw:record>
      <srw:extraRequestData>
        <authenticationToken>{username}/{password}</authenticationToken>
      </srw:extraRequestData>
    </ucp:updateRequest>
  </soap:Body>
</soap:Envelope>"""


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def strip_ns(tag: str) -> str:
    """Removes XML namespace prefix from element tags."""
    return tag.split("}")[-1] if "}" in tag else tag


def parse_marc_content(file_path: str) -> str:
    """Parses an XML file and extracts MARC21 fields cleanly without namespace prefixes."""
    tree = ET.parse(file_path)
    root = tree.getroot()

    record_node = None
    for elem in root.iter():
        if elem.tag.endswith("record") and "MARC" in elem.tag:
            record_node = elem
            break
    if record_node is None and root.tag.endswith("record"):
        record_node = root

    if record_node is None:
        raise ValueError(f"No MARC record element found in {file_path}")

    marc_lines = []
    for child in record_node:
        tag_name = strip_ns(child.tag)
        if tag_name not in ["leader", "controlfield", "datafield"]:
            continue

        attrs = "".join([f' {k}="{v}"' for k, v in child.attrib.items() if "}" not in k])

        if tag_name == "leader":
            marc_lines.append(f"            <{tag_name}>{child.text}</{tag_name}>")
        else:
            if len(child) == 0 and child.text:
                marc_lines.append(f"            <{tag_name}{attrs}>{child.text}</{tag_name}>")
            else:
                marc_lines.append(f"            <{tag_name}{attrs}>")
                for sub in child:
                    sub_tag = strip_ns(sub.tag)
                    sub_attrs = "".join([f' {k}="{v}"' for k, v in sub.attrib.items() if "}" not in k])
                    marc_lines.append(f"              <{sub_tag}{sub_attrs}>{sub.text}</{sub_tag}>")
                marc_lines.append(f"            </{tag_name}>")

    return "\n".join(marc_lines)


def send_sru_request(file_path: str, action: str, url: str, username: str, password: str):
    """Formats payload and sends the update/create request to the SRU server."""
    action_uri = (
        "info:srw/action/1/create" if action.lower() == "create" else "info:srw/action/1/replace"
    )

    marc_content_str = parse_marc_content(file_path)

    payload = SOAP_TEMPLATE.format(
        action=action_uri,
        marc_content=marc_content_str,
        username=username,
        password=password,
    )

    headers = {"Content-Type": "text/xml; charset=utf-8"}
    print(f"Sending SRU [{action.upper()}] request for: {os.path.basename(file_path)}...\n")

    try:
        response = requests.post(
            url,
            data=payload.encode("utf-8"),
            headers=headers,
            auth=HTTPBasicAuth(username, password),
            timeout=30,
        )

        if response.status_code == 200:
            namespaces = {
                "ucp": "http://www.loc.gov/zing/srw/update/",
                "srw": "http://www.loc.gov/zing/srw/",
                "diag": "http://www.loc.gov/zing/srw/diagnostic/",
            }
            res_root = ET.fromstring(response.text)
            status = res_root.find(".//ucp:operationStatus", namespaces)

            if status is not None and status.text == "success":
                ppn_elem = res_root.find(".//ucp:recordIdentifier", namespaces)
                ppn = ppn_elem.text if ppn_elem is not None else "Unknown"
                print(f"-> Success! Record processed. ID/PPN: {ppn}")
            else:
                diag_msg = res_root.find(".//diag:message", namespaces)
                msg_text = diag_msg.text if diag_msg is not None else "Validation error without details"
                print(f"-> SRU Error: {msg_text}")
        else:
            print(f"-> Server Error (Status {response.status_code}):")
            print(response.text)

    except Exception as e:
        print(f"-> Request failed: {e}")


# ==========================================
# 3. CONFIGURATION & EXECUTION
# ==========================================
SRU_URL = "https://devel.dnb.de/sru_ru/"
FILE_PATH = "data/C004fd089.xml"
ACTION = "create"

USERNAME = os.getenv("SRU_USERNAME") or input("Enter SRU Username: ")
PASSWORD = os.getenv("SRU_PASSWORD") or getpass.getpass("Enter SRU Password: ")

send_sru_request(
    file_path=FILE_PATH,
    action=ACTION,
    url=SRU_URL,
    username=USERNAME,
    password=PASSWORD
)
